# 81 · Published human vs mouse kidney findings, re-read along the PT

The published comparisons of human and mouse kidney are level differences inside one segment (CellSpectra),
whole-tissue differences (Raghubar), functional single-gene contrasts (Fernandes, Basit) or sex programs (Xiong).
None asks where along S1 → S3 a published program sits in each species. This notebook takes their claims and gene
lists, frozen before any computation (`docs/results/published-findings/`), and asks three things of each:

| Part | Question | Status |
|---|---|---|
| Reproduce | Does the published direction hold in all four human × mouse specimen pairs? | level; probe-exposed; descriptive |
| Place | Does the difference change between S1, S2 and S3, and where is it largest? | between segments; calibrated |
| Describe | Where does a program switch along the axis, how sharply, and is it the same place in human and mouse? | continuous; descriptive |

**Protocol:** `docs/results/pt-published-findings.md` (commit 0a44dec), frozen before this notebook ran. Notebook 82
asks the same questions of the papers' own data (Cheval 2012) and of the single-nucleus atlases.

**Cohort.** Two male control mice and two cortex sections of one male human donor. Specimens are the replicates;
everything is descriptive. Nothing here refits a gene model: the curves are notebook 71's, the statistics notebook
75's and the pseudobulks notebook 79's rule.

## 1 · Setup

In [ ]:
import argparse
import hashlib
import json
import os
import sys
import time
from pathlib import Path

for _var in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ.setdefault(_var, os.environ.get('PSEUDOSPACE_N_JOBS', '24'))   # thread cap; never in a cache key
NOTEBOOK_START = time.perf_counter()
starts = [Path.cwd()] + ([Path(__file__).resolve().parent] if '__file__' in globals() else [])
project = next(p for s in starts for p in (s, *s.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
EXPORT = results_root / 'package_pseudospace' / 'hvm' / 'pt_units.h5ad'
upstream13 = results_root / 'minimal_pt_scfates'                  # the ortholog map
T71 = results_root / 'pt_pathways_package_pseudospace'            # read only: tables and stage cache
T72 = results_root / 'pt_package_coordinate_validation'
T75 = results_root / 'pt_package_pathway_methods'
T79 = results_root / 'pt_known_species_programs'
nb38 = results_root / 'pt_literature_deep'
LRSZ_FILE = results_root / 'pt_reconstruction_v3' / 'coordinates' / 'lrs_z.csv'   # notebook 53's zone-landmark ratio
KNOWN_DIR = project / 'docs' / 'results' / 'known-programs'        # notebook 79's frozen programs
LIT_DIR = project / 'docs' / 'results' / 'published-findings'      # this protocol's frozen lists
XIONG_DIR = data_root / 'external' / 'xiong2023_sex_bias'
output = results_root / 'pt_published_findings'
tables, figures = output / 'tables', output / 'figures'
for folder in (tables, figures):
    folder.mkdir(parents=True, exist_ok=True)
DS = 'human_vs_mouse'
PARTITION_FILES = {'condition': 'species', 'relabeling 1': 'swap_HUK1_COR1_Ctrl1A2', 'relabeling 2': 'swap_HUK1_COR1_Ctrl1A4'}
SHA = {KNOWN_DIR / 'programs_hvm.csv': 'be61b8183c51d798ee33314e6226d5d8dd269188b82eb60ae65c888a355db79e',
       KNOWN_DIR / 'programs_hvm_members.csv': '2a3889075ca4130b102661409aee1390f1dd1f936a4171c9c9b7b0a7365299bf',
       LIT_DIR / 'programs_lit.csv': 'feeee7503b3b522df5876b01c17855ff0e1bcd39430867cec7f2f623043a6e4e',
       LIT_DIR / 'programs_lit_members.csv': '8bcbd2556f52b859ad4ef62ffd2c98c14d4821e784a68ec979ecfbebbab219c7',
       LIT_DIR / 'genes_lit.csv': '111cca20e92708f7fbceccca6b5fa0d49453544bfb5a57c26336e8e3805a22d8',
       XIONG_DIR / 'PT-S1_fWTvsmWT.csv': '4764f77435e1b232e2e686770776c16b3c9f93a085eee1b52d9914b116f00eee',
       XIONG_DIR / 'PT-S2_fWTvsmWT.csv': '59e9103e6210a43d6569166d8d3e40b3ec1e089e3caadcbe4244bd8a23c220a6',
       XIONG_DIR / 'PT-S3_fWTvsmWT.csv': '966286c7ae61b2f78db9b6f3aa4fe3d9c942d67527b0ca309f96ed8ded9cd9cd'}
REQUIRED = [EXPORT, upstream13 / 'ortholog_map_used.csv', nb38 / 'probe_counts.csv', LRSZ_FILE, *SHA,
            *[T71 / 'tables' / f for f in ('inputs.csv', f'universe_{DS}.csv', f'gene_sets_{DS}.json', f'gene_statistics_{DS}.csv')],
            T72 / 'tables' / 'transitions.csv', T79 / 'tables' / 'calibration.csv', T79 / 'tables' / 'pseudobulk_rates.csv.gz',
            *[T75 / 'tables' / f for f in (f'gene_statistics_segment_{DS}.csv', f'pathway_tests_segment_{DS}.csv')]]
for path in REQUIRED:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path} (run notebooks 70–72, 75 and 79 first)')

NOTEBOOK_LOGIC_VERSION = '81.published_findings.1'      # Bump when a calculation changes (no stage cache here).
FINGERPRINT = 'efafc9ac2e99'                            # confirmed hvm pass-1 labels (notebook 70)
ALPHA, CAMERA_COR, CALIBRATION_MAX = .05, .01, .075     # project set rule; cameraPR; 75's swap rule
EFFECT = .25                                            # log2; 76's O5 threshold
MIN_MEMBERS, FLOOR, MIN_DETECTION = 5, .5, .05
REGISTRATION_GAP = .37                                  # 72's species registration gap (segment units)
SEGMENTS = ('S1', 'S2', 'S3')
SCOPES = {'S1': ['S1'], 'S2': ['S2'], 'S3': ['S3'], 'PT': None}
PARTS = ['condition', 'relabeling 1', 'relabeling 2']
NAMES = ['Ctrl1A2', 'Ctrl1A4', 'HUK1_COR1', 'HUK1_MED1']
MOUSE, HUMAN = NAMES[:2], NAMES[2:]
CASE, REF = ('HUK1_COR1', 'HUK1_MED1'), ('Ctrl1A2', 'Ctrl1A4')
PAIRS = [f'{c}−{r}' for c in CASE for r in REF]
SPECIES_OF = {**{s: 'mouse' for s in MOUSE}, **{s: 'human' for s in HUMAN}}
LABEL_ORIENTATION = {'Cyp2e1', 'Cyp7b1', 'Gatm', 'Slc7a13', 'Slc13a3', 'Slc22a6', 'Slc5a2', 'Slc22a7', 'Slc5a12',
                     'Aqp2', 'Aqp4', 'Atp6v0d2', 'Rhcg', 'Slc14a2', 'Wnt7b'}
LIBRARY_ORDER = ['MSigDB_Hallmark_2020', 'KEGG_2019_Mouse', 'Reactome_2022']   # protocol L: ties by this order

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

MM = 1 / 25.4
INK, MUTED, LIGHT = '#0b0b0b', '#52514e', '#d9d8d4'
MOUSE_COLOR, HUMAN_COLOR, RELABEL_COLOR = '#0072B2', '#D55E00', '#9a9a9a'
SPEC_COLOR = {'Ctrl1A2': '#0072B2', 'Ctrl1A4': '#56B4E9', 'HUK1_COR1': '#D55E00', 'HUK1_MED1': '#E69F00'}
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5, 'axes.titleweight': 'bold',
    'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6, 'legend.fontsize': 6, 'legend.frameon': False,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.linewidth': .6, 'lines.linewidth': 1.0,
    'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_colwidth', 90, 'display.max_rows', 160)
READING, GUARDS = [], []


def save(fig, stem):
    for suffix in ('png', 'pdf'):
        fig.savefig(figures / f'{stem}.{suffix}', bbox_inches='tight')
    plt.show()
    plt.close(fig)


def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


print('Results folder:', output, '· logic', NOTEBOOK_LOGIC_VERSION)

## 2 · Inputs and guards

The inputs are built as notebooks 71 and 79 build them (`package_inputs.hvm_inputs`; the 13 `mt-` genes are out).
Guards (not cached):
- **G1/G2:** the export contract, the confirmed pass-1 fingerprint, and 71's universe, sets and units.
- **G3:** 71's cached condition model reproduces its T_spatial and moderated-F z.
- **G4:** 75's segment table covers the universe.
- **G5:** every frozen file matches its sha256; members lie in the universe; none is a label or orientation gene.
- **G6:** the logistic transitions reproduce 72's.
- **G7:** the pseudobulks and the calibration table reproduce notebook 79's.

In [ ]:
from pseudospace.continuous_descriptors import logistic_transitions, specimen_curves
from pseudospace.discrete_comparators import swap_rule
from pseudospace.package_inputs import hvm_inputs, read_pt_units
from pseudospace.pathway_pipelines import camera_pr, common_grid, moderated_f, partition_groups
from pseudospace.stats_gam import gam_internal_knots
from pseudospace import program_localization as pl
from pseudospace import published_findings as pf
from pseudospace import species_calls as sc

t0 = time.perf_counter()
orthologs = pd.read_csv(upstream13 / 'ortholog_map_used.csv')
MT = sorted(g for g in orthologs.mouse_symbol.astype(str) if g.startswith('mt-'))
assert len(MT) == 13, MT
UNITS = read_pt_units(EXPORT, 'hvm')                                                       # G1
PROV = UNITS.uns['provenance']
assert str(PROV['notebook_logic_version']).startswith('70.'), PROV['notebook_logic_version']
assert PROV['fingerprints']['pass1']['membership_sha1'] == FINGERPRINT, PROV['fingerprints']['pass1']
panels = PROV['config']['marker_panels_yaml']
label_orientation = ({str(g) for s in ('PT-S1', 'PT-S2', 'PT-S3') for g in panels['label_panels'][s]}
                     | {str(g) for side in ('early', 'late') for g in panels['axis_anchors'][side]})
assert label_orientation == LABEL_ORIENTATION, label_orientation ^ LABEL_ORIENTATION
INP = hvm_inputs(UNITS, data_root, orthologs, exclude_genes=MT)
GENES = list(INP['genes'])
SETS = INP['gene_sets']
universe71 = pd.read_csv(T71 / 'tables' / f'universe_{DS}.csv')
universe71 = universe71[universe71.eligible.astype(bool)]
assert GENES == universe71.gene.astype(str).tolist(), 'G2: the universe differs from notebook 71'
assert SETS == json.loads((T71 / 'tables' / f'gene_sets_{DS}.json').read_text()), 'G2: the gene sets differ from notebook 71'
assert int(pd.read_csv(T71 / 'tables' / 'inputs.csv', index_col=0).loc['units in the common support', 'Human vs mouse PT']) == len(INP['position'])
POSITION = np.asarray(INP['position'], float)
SPECIMEN = INP['specimen'].astype(str)
SEGMENT = pd.Series(INP['segment']).str.replace('PT-', '', regex=False).to_numpy()
COUNTS = INP['counts'].tocsr()
LIBRARY = np.asarray(INP['library'], float)
LOG_MEAN_COUNT = np.log(np.asarray(COUNTS.mean(axis=0)).ravel())
OBS = INP['obs']
labels = list(partition_groups(SPECIMEN, INP['case']))
assert labels == ['species', 'swap:HUK1_COR1+Ctrl1A2', 'swap:HUK1_COR1+Ctrl1A4'], labels
GUARDS.append(f'G1/G2: export contract; pass-1 fingerprint {FINGERPRINT}; universe ({len(GENES):,}), sets ({len(SETS):,}) and '
              f'units ({len(POSITION):,}) equal notebook 71; the label/orientation genes equal the protocol\'s 15')


def stored(folder, stage):
    """A notebook's newest cached payload of one stage (read only); raises when its cache was purged."""
    files = sorted((folder / 'stage_cache').glob(f'{stage}__*.npz'),
                   key=lambda p: json.loads(Path(str(p) + '.key.json').read_text())['written_utc'])
    if not files:
        raise FileNotFoundError(f'{folder.name}: no cached {stage}')
    with np.load(files[-1], allow_pickle=False) as payload:
        return {k: payload[k] for k in payload.files}


STAT71 = pd.read_csv(T71 / 'tables' / f'gene_statistics_{DS}.csv', header=[0, 1], index_col=0)
assert STAT71.index.astype(str).tolist() == GENES
MODELS = {part: stored(T71, f'gene_model_{DS}_{f}') for part, f in PARTITION_FILES.items()}
np.testing.assert_allclose(MODELS['condition']['stat|T_spatial'], STAT71['condition'].T_spatial.to_numpy(), rtol=1e-12, atol=0)
again = moderated_f(STAT71['condition'].T_spatial, 6, STAT71['condition'].LR_spec.to_numpy(), 12,
                    covariate=LOG_MEAN_COUNT, robust=False)
np.testing.assert_allclose(again.z.to_numpy(), STAT71['condition'].z.to_numpy(), rtol=1e-9, atol=1e-9)
GRID = MODELS['condition']['grid']
np.testing.assert_allclose(GRID, common_grid(POSITION, SPECIMEN, 101), rtol=0, atol=1e-12)
Z_POS = {part: STAT71[part].z.astype(float) for part in PARTS}
GUARDS.append("G3: notebook 71's cached condition model reproduces its T_spatial and moderated-F z")
SEG75 = pd.read_csv(T75 / 'tables' / f'gene_statistics_segment_{DS}.csv', header=[0, 1], index_col=0)
assert SEG75.index.astype(str).tolist() == GENES                                            # G4
Z_SEG = {part: SEG75[part].z.astype(float) for part in PARTS}
GUARDS.append("G4: notebook 75's segment table covers the universe")

for path, digest in SHA.items():                                                            # G5
    assert sha256(path) == digest, f'G5: {path.name} differs from its frozen sha256'
P79, META79 = pl.read_programs(KNOWN_DIR / 'programs_hvm.csv', GENES, LABEL_ORIENTATION, sha256=SHA[KNOWN_DIR / 'programs_hvm.csv'])
PLIT, METALIT = pl.read_programs(LIT_DIR / 'programs_lit.csv', GENES, LABEL_ORIENTATION, sha256=SHA[LIT_DIR / 'programs_lit.csv'])
MEMBERS_LIT = pd.read_csv(LIT_DIR / 'programs_lit_members.csv')
assert all(MEMBERS_LIT[MEMBERS_LIT.program_id.eq(p)].gene.tolist() == v for p, v in PLIT.items())
GENES_LIT = pd.read_csv(LIT_DIR / 'genes_lit.csv')
GUARDS.append(f'G5: {len(SHA)} frozen files match their sha256; {len(P79)} notebook 79 programs and {len(PLIT)} new '
              'programs, every member in the universe, no label/orientation gene')

TRANSITIONS = pd.read_csv(T72 / 'tables' / 'transitions.csv')
TRANSITIONS = TRANSITIONS[TRANSITIONS.dataset.eq('hvm')].reset_index(drop=True)
again_t = logistic_transitions(POSITION, INP['segment'], SPECIMEN).merge(TRANSITIONS, on=['specimen', 'transition'])
assert len(again_t) == 8 and np.allclose(again_t.position_x, again_t.position_y, atol=1e-6)   # G6
CUTS = sc.mean_transitions(TRANSITIONS, NAMES)
S_GRID = sc.to_segment_units(GRID, CUTS)
GUARDS.append(f"G6: logistic transitions reproduce 72's; pooled cuts S1→S2 {CUTS[0]:.3f}, S2→S3 {CUTS[1]:.3f}")
probes = pd.read_csv(nb38 / 'probe_counts.csv').drop_duplicates('mouse_symbol').set_index('mouse_symbol').reindex(GENES)
print('\n'.join('· ' + g for g in GUARDS))
print(f'Inputs {time.perf_counter() - t0:.0f} s · {len(POSITION):,} units · {len(GENES):,} genes · {len(SETS):,} sets')

## 3 · Per-gene statistics (notebook 79's rules)

- **Pseudobulks:** log2((Σ counts + 0.5) / Σ mt-free library) per specimen × scope (S1, S2, S3, PT).
- **Contrasts:** the human − mouse group effect, the 4 pairs and the 2 relabelings, each centred by its median gene.
- **Moderated level t** per scope (squeezeVar on the log mean count per unit).
- **Zonation:** each species' segment-vs-other-segments contrast per specimen, centred, with its moderated t.
- **Calibration** of each statistic on 71's 1,513 sets with 75's swap rule; a failing statistic's program p becomes
  the empirical p against the relabelings. G7: the pseudobulks and calibration equal notebook 79's.

In [ ]:
def pseudobulks(counts, library, specimen, segment):
    return pl.segment_pseudobulk(counts, library, specimen, segment, SCOPES, genes=GENES, floor=FLOOR)


def level_statistics(rates, covariate):
    stats, pairs = {}, {}
    for scope in rates:
        for part in PARTS:
            eff = pl.partition_effects(rates[scope], CASE, REF, part)
            stats[(scope, part)] = pl.moderated_t(eff.effect, eff.s2, float(eff.df.iloc[0]), covariate=covariate)
        pairs[scope] = pd.DataFrame(pl.pair_contrasts(rates[scope], CASE, REF))
    return stats, pairs


def zonation_statistics(rates, covariate):
    out = {}
    for species, specimens in (('mouse', MOUSE), ('human', HUMAN)):
        for seg in SEGMENTS:
            con = pl.segment_contrast({s: rates[s] for s in SEGMENTS}, specimens, seg)
            t = pl.moderated_t(con.effect, con.s2, 1., covariate=covariate)
            out[(species, seg)] = t.join(con[specimens])
    return out


t0 = time.perf_counter()
RATES = pseudobulks(COUNTS, LIBRARY, SPECIMEN, SEGMENT)
saved79 = pd.read_csv(T79 / 'tables' / 'pseudobulk_rates.csv.gz', header=[0, 1], index_col=0)
for s in SCOPES:                                                                             # G7a
    np.testing.assert_allclose(pd.DataFrame(RATES[s]).reindex(columns=NAMES).to_numpy(),
                               saved79[s].reindex(index=GENES, columns=NAMES).to_numpy(), rtol=1e-10, atol=1e-10)
LEVEL, PAIR_CON = level_statistics(RATES, LOG_MEAN_COUNT)
ZON = zonation_statistics(RATES, LOG_MEAN_COUNT)
CONTRASTS = {'condition': pd.DataFrame({s: LEVEL[(s, 'condition')].effect for s in SCOPES}),
             **{part: pd.DataFrame({s: LEVEL[(s, part)].effect for s in SCOPES}) for part in PARTS[1:]},
             **{pair: pd.DataFrame({s: PAIR_CON[s][pair] for s in SCOPES}) for pair in PAIRS}}
SEG_CON = {spec: pd.DataFrame({seg: ZON[(SPECIES_OF[spec], seg)][spec] for seg in SEGMENTS}) for spec in NAMES}
DETECTION = {spec: pd.Series(np.asarray((COUNTS[SPECIMEN == spec] > 0).mean(axis=0)).ravel(), index=GENES) for spec in NAMES}


def calibration_runs(statistics, name, two_sided):
    runs = {part: camera_pr(statistics[part], SETS, inter_gene_cor=CAMERA_COR).set_index('pathway_id') for part in PARTS}
    if two_sided:
        calls = {part: int((runs[part].fdr <= ALPHA).sum()) for part in PARTS}
        null = np.concatenate([runs[p].p.to_numpy(float) for p in PARTS[1:]])
    else:
        calls = {part: int(runs[part].call.sum()) for part in PARTS}
        null = np.concatenate([runs[p].p_one_sided.to_numpy(float) for p in PARTS[1:]])
    verdict = swap_rule(calls['condition'], [calls[p] for p in PARTS[1:]], len(runs['condition']), null)
    return {'statistic': name, 'tested': len(runs['condition']), **{f'calls {p}': calls[p] for p in PARTS}, **verdict}, null


CALIB, NULLS = [], {}
for scope in SCOPES:
    row, NULLS[('level', scope)] = calibration_runs({p: LEVEL[(scope, p)].t for p in PARTS}, f'level t · {scope}', True)
    CALIB.append(row)
row, NULLS['z_seg'] = calibration_runs(Z_SEG, 'segment F z (75)', False)
CALIB.append(row)
row, NULLS['z_pos'] = calibration_runs(Z_POS, 'positional F z (71)', False)
CALIB.append(row)
CALIBRATION = pd.DataFrame(CALIB).set_index('statistic')
saved_cal = pd.read_csv(T79 / 'tables' / 'calibration.csv', index_col=0)
np.testing.assert_allclose(CALIBRATION['calibration share'], saved_cal['calibration share'].reindex(CALIBRATION.index), rtol=1e-10)
PASSES = {name: CALIBRATION.loc[name, 'calibration'] == 'pass' for name in CALIBRATION.index}
GUARDS.append("G7: the pseudobulks and the calibration table reproduce notebook 79's")
CALIBRATION.to_csv(tables / 'calibration.csv')
display(CALIBRATION[['calls condition', 'calls relabeling 1', 'calls relabeling 2', 'calibration share', 'calibration']].round(3))
print(f'Per-gene statistics and calibration {time.perf_counter() - t0:.0f} s')

## 4 · Program tests and classes (notebook 79's rules, generalised to any sign and scope)

In [ ]:
from scipy.stats import false_discovery_control


def bh(p):
    p = pd.Series(p, dtype=float)
    out = pd.Series(np.nan, index=p.index)
    ok = p.notna()
    if ok.any():
        out[ok] = false_discovery_control(p[ok].to_numpy())
    return out


def with_calibration(res, null, passes, two_sided):
    res = res.copy()
    if passes:
        res['p_claimed_cal'], res['p_opposite_cal'] = res.p_claimed, res.p_opposite
        return res
    if two_sided:
        emp = pl.calibrated(res.p.to_numpy(float), null)
        matches = np.isclose(res.p_claimed, res.p / 2)
        res['p_claimed_cal'] = np.where(matches, emp / 2, 1 - emp / 2)
        res['p_opposite_cal'] = np.where(matches, 1 - emp / 2, emp / 2)
    else:
        res['p_claimed_cal'] = pl.calibrated(res.p_claimed.to_numpy(float), null)
        res['p_opposite_cal'] = np.nan
    return res


def level_tests(programs, scopes, signs):
    """Replication on the level t at each program's scope (condition calibrated, relabelings nominal); BH over all."""
    rows = []
    for scope in sorted(set(scopes.values())):
        progs = {p: programs[p] for p in programs if scopes[p] == scope}
        if not progs:
            continue
        stat = {part: LEVEL[(scope, part)].t for part in PARTS}
        res = pl.program_test(stat['condition'], progs, {p: signs[p] for p in progs}, inter_gene_cor=CAMERA_COR)
        res = with_calibration(res, NULLS[('level', scope)], PASSES[f'level t · {scope}'], True)
        for part in PARTS[1:]:
            res[f'{part} p_claimed'] = pl.program_test(stat[part], progs, {p: signs[p] for p in progs},
                                                       inter_gene_cor=CAMERA_COR).p_claimed
        res['scope'] = scope
        rows.append(res)
    out = pd.concat(rows).reindex(list(programs))
    out['q_claimed'], out['q_opposite'] = bh(out.p_claimed_cal), bh(out.p_opposite_cal)
    out['specific'] = (out['relabeling 1 p_claimed'] > ALPHA) & (out['relabeling 2 p_claimed'] > ALPHA)
    return out


def between_tests(programs):
    """Between-segment (segment F z, Up) and along-the-axis (positional F z, Up; sensitivity) tests; BH over the set."""
    out = {}
    for name, stat, key in (('between', Z_SEG, 'segment F z (75)'), ('axis', Z_POS, 'positional F z (71)')):
        res = pl.program_test(stat['condition'], programs, inter_gene_cor=CAMERA_COR)
        res = with_calibration(res, NULLS['z_seg' if name == 'between' else 'z_pos'], PASSES[key], False)
        out[f'{name} p'] = res.p_claimed_cal
        for part in PARTS[1:]:
            out[f'{name} {part} p'] = pl.program_test(stat[part], programs, inter_gene_cor=CAMERA_COR).p_claimed
    out = pd.DataFrame(out)
    out['between q'], out['axis q'] = bh(out['between p']), bh(out['axis p'])
    return out


def summarise(programs, stat='median'):
    long = pl.segment_summary(CONTRASTS, programs, min_members=MIN_MEMBERS, stat=stat)
    return long.pivot_table(index=['program', 'contrast'], columns='scope', values='value')


def classify(programs, summary, between, signs, scopes=None, tests=None):
    """Localization (79 §5.3) of the human − mouse difference; replication (79 §5.1) when level tests are given."""
    rows = []
    for p in programs:
        d = signs[p]
        cond = summary.loc[(p, 'condition'), list(SEGMENTS)].to_numpy(float)
        q_pairs = np.array([summary.loc[(p, pair), list(SEGMENTS)].to_numpy(float) for pair in PAIRS]) * d
        q_relab = np.array([summary.loc[(p, part), list(SEGMENTS)].to_numpy(float) for part in PARTS[1:]]) * d
        loc = pl.localization_class(between.loc[p, 'between q'], cond * d, q_pairs, q_relab)
        row = {'program': p, 'd': d, **{f'P {s}': cond[k] for k, s in enumerate(SEGMENTS)},
               'localization': loc['class'], 'peak': loc['peak'], 'margin': loc['margin'],
               'pairs unanimous': loc['unanimous'], 'relabeled ratio 1': loc['ratios'][0],
               'relabeled ratio 2': loc['ratios'][1], 'present in': ', '.join(loc['present']),
               'no difference resolved in': ', '.join(loc['unresolved']), 'between q': between.loc[p, 'between q'],
               'axis q': between.loc[p, 'axis q']}
        if tests is not None:
            scope = scopes[p]
            at = summary.loc[(p, 'condition'), scope] * d
            pairs_at = np.array([summary.loc[(p, pair), scope] for pair in PAIRS]) * d
            row.update({'scope': scope, 'Q at scope': at, 'pairs with claimed sign': int((pairs_at > 0).sum()),
                        'q claimed': tests.loc[p, 'q_claimed'], 'specific': bool(tests.loc[p, 'specific']),
                        'replication': pl.replication_class(tests.loc[p, 'q_claimed'], bool(tests.loc[p, 'specific']),
                                                            at, pairs_at, tests.loc[p, 'q_opposite'])})
        rows.append(row)
    return pd.DataFrame(rows).set_index('program')


def family(fam):
    return {p: PLIT[p] for p in METALIT.index if METALIT.loc[p, 'family'] == fam}


SIGN_LIT = {p: (1 if str(METALIT.loc[p, 'direction']).startswith('human') else -1) for p in METALIT.index}
SCOPE_LIT = {p: str(METALIT.loc[p, 'segment']) for p in METALIT.index}
print({fam: len(family(fam)) for fam in ('A1', 'A3', 'C1')})

## 5 · A1 · Are CellSpectra's segment-specific species genes segment-specific?

U-Hk / U-Mk hold the members of CellSpectra's human-high / mouse-high block of segment k that no other segment's
block of the same direction lists; U-HC / U-MC hold the 40 genes listed in all three. The listing implies a
difference specific to Sk (unique) or PT-wide (core). Readout (protocol A1):
- **supported**: localized to Sk, or graded with its condition peak in Sk;
- **not supported (PT-wide)**: uniform within resolution, or the peak in another segment by ≥ 0.25;
- otherwise **not resolved**.

In [ ]:
A1 = family('A1')
A1_SUMMARY = summarise(A1)
A1_BETWEEN = between_tests(A1)
A1_TESTS = level_tests(A1, SCOPE_LIT, SIGN_LIT)
A1_CLASSES = classify(A1, A1_SUMMARY, A1_BETWEEN, SIGN_LIT, SCOPE_LIT, A1_TESTS)


def a1_readout(p, r):
    listed = SCOPE_LIT[p]
    if listed == 'PT':
        return 'core (expected uniform)'
    if r.localization == f'localized to {listed}' or (r.localization == 'graded' and r.peak == listed):
        return 'supported (segment-specific)'
    if r.localization == 'uniform within resolution' or (r.peak != listed and r.margin >= EFFECT):
        return 'not supported (PT-wide or elsewhere)'
    return 'not resolved'


A1_CLASSES['n'] = [len(A1[p]) for p in A1_CLASSES.index]
A1_CLASSES['listed'] = [SCOPE_LIT[p] for p in A1_CLASSES.index]
A1_CLASSES['readout'] = [a1_readout(p, r) for p, r in A1_CLASSES.iterrows()]
display(A1_CLASSES[['n', 'listed', 'P S1', 'P S2', 'P S3', 'between q', 'localization', 'peak', 'margin',
                    'relabeled ratio 1', 'relabeled ratio 2', 'replication', 'readout']].round(3))

## 6 · A2 · Gene-level classes of every CellSpectra block member (descriptive)

Per member of S-H1 … S-M3 (79's blocks), with Q_j = d · P_j in the condition and in each of the 4 pairs: present in
all segments / restricted to the listed segment / elsewhere only / opposite somewhere / not reproduced (the
protocol's order). Subsets: unique (A1), core, or listed in two segments.

In [ ]:
S_BLOCKS = [p for p in META79.index if META79.loc[p, 'family'] == 'species']
gene_rows = []
for p in S_BLOCKS:
    d = 1 if str(META79.loc[p, 'direction']).startswith('human') else -1
    k = SEGMENTS.index(META79.loc[p, 'segment'])
    unique, core = set(PLIT[f'U-{p[2]}{p[3]}']), set(PLIT[f'U-{p[2]}C'])
    for g in P79[p]:
        qc = d * CONTRASTS['condition'].loc[g, list(SEGMENTS)].to_numpy(float)
        qp = d * np.array([CONTRASTS[pair].loc[g, list(SEGMENTS)].to_numpy(float) for pair in PAIRS])
        gene_rows.append({'block': p, 'gene': g, 'subset': 'unique' if g in unique else 'core' if g in core else 'two segments',
                          'class': pf.gene_class(qc, qp, k), **{f'Q {s}': qc[j] for j, s in enumerate(SEGMENTS)},
                          'equal probes': bool(probes.loc[g, 'mouse_probes'] == probes.loc[g, 'human_probes'])})
GENE_CLASSES = pd.DataFrame(gene_rows)
A2_COUNTS = GENE_CLASSES.pivot_table(index=['block', 'subset'], columns='class', values='gene', aggfunc='count', fill_value=0)
display(A2_COUNTS)
display(GENE_CLASSES.groupby('class').size().rename('genes (all blocks)'))

## 7 · A3 · Do CellSpectra's conserved PT functions keep their place along the PT?

Six GO-defined functions CellSpectra names as conserved (or, for LDL particle binding, least coordinated) in PT.
Placement within each species: the program median of each specimen's centred segment contrast and its peak segment.
Species × segment: 79's localization class (d = the sign of the condition P where |P| is largest).
- **placement conserved**: the same peak segment in all 4 specimens and localization *no difference* or *uniform*;
- **placement differs**: localization *localized* or *graded*, or the species' peak segments differ in both specimens;
- otherwise **not resolved**.

In [ ]:
A3 = family('A3')
A3_SUMMARY = summarise(A3)
A3_SIGN = {}
for p in A3:
    cond = A3_SUMMARY.loc[(p, 'condition'), list(SEGMENTS)].to_numpy(float)
    A3_SIGN[p] = 1 if cond[int(np.argmax(np.abs(cond)))] >= 0 else -1
A3_BETWEEN = between_tests(A3)
A3_CLASSES = classify(A3, A3_SUMMARY, A3_BETWEEN, A3_SIGN)


def specimen_placement(members):
    """{specimen: program median of the centred segment contrasts (S1, S2, S3)} and the peak segment per specimen."""
    vals = {spec: SEG_CON[spec].loc[members].median() for spec in NAMES}
    return pd.DataFrame(vals).T, {spec: SEGMENTS[int(np.argmax(v.to_numpy()))] for spec, v in vals.items()}


rows = []
for p in A3:
    table, peaks = specimen_placement(A3[p])
    r = A3_CLASSES.loc[p]
    mouse_peaks, human_peaks = {peaks[s] for s in MOUSE}, {peaks[s] for s in HUMAN}
    same_all = len(set(peaks.values())) == 1
    differs_species = len(mouse_peaks) == 1 and len(human_peaks) == 1 and mouse_peaks != human_peaks
    if same_all and r.localization in ('no difference', 'uniform within resolution'):
        readout = 'placement conserved'
    elif r.localization.startswith('localized') or r.localization == 'graded' or differs_species:
        readout = 'placement differs'
    else:
        readout = 'not resolved'
    rows.append({'program': p, 'function': METALIT.loc[p, 'published_claim'].split(':')[0], 'n': len(A3[p]),
                 **{f'peak {s}': peaks[s] for s in NAMES},
                 **{f'{s} {seg}': table.loc[s, seg] for s in NAMES for seg in SEGMENTS},
                 'localization': r.localization, 'between q': r['between q'], 'readout': readout})
A3_TABLE = pd.DataFrame(rows).set_index('program')
display(A3_TABLE[['function', 'n'] + [f'peak {s}' for s in NAMES] + ['localization', 'between q', 'readout']].round(4))

## 8 · C1 · Raghubar's whole-cortex species difference: PT-intrinsic, and uniform along the PT?

R-ENERGY and R-METAB (mouse-high) and R-STRUCT (human-high) from their GO tables and DE list. Replication at the PT
scope (level, calibrated empirically, descriptive), the localization class, and for R-STRUCT the share of members
detected in at least 10% of units in both human sections against the universe's share.

In [ ]:
C1 = family('C1')
C1_SCOPE = {p: 'PT' for p in C1}
C1_SUMMARY = summarise(C1)
C1_TESTS = level_tests(C1, C1_SCOPE, SIGN_LIT)
C1_BETWEEN = between_tests(C1)
C1_CLASSES = classify(C1, C1_SUMMARY, C1_BETWEEN, SIGN_LIT, C1_SCOPE, C1_TESTS)
detected_both_human = (DETECTION['HUK1_COR1'] >= .1) & (DETECTION['HUK1_MED1'] >= .1)
C1_CLASSES['detected ≥10% in both human sections'] = [float(detected_both_human[C1[p]].mean()) for p in C1_CLASSES.index]
C1_CLASSES['universe share detected'] = float(detected_both_human.mean())
display(C1_CLASSES[['d', 'P S1', 'P S2', 'P S3', 'Q at scope', 'pairs with claimed sign', 'q claimed', 'specific',
                    'replication', 'localization', 'peak', 'margin', 'between q',
                    'detected ≥10% in both human sections', 'universe share detected']].round(4))

## 9 · Gene-level hypotheses (C2 Raghubar, D Fernandes, F Basit; single genes, descriptive)

Per gene: detection per specimen, the centred segment contrasts per specimen (S3 against the other two, and S1/S2
against S3), and the centred human − mouse PT difference in each pair. Probe counts are listed: level rules are
probe-exposed. Each prediction in `genes_lit.csv` is translated to the protocol's rule and marked consistent or not.

In [ ]:
TRANSPORTERS = [g for g in GENES if g.startswith(('Slc', 'Abc'))]
pt_pairs = pd.DataFrame({pair: PAIR_CON['PT'][pair] for pair in PAIRS})
universe_detection = {spec: float(DETECTION[spec].median()) for spec in NAMES}


def s3_vs_early(spec, g):
    """Centred r_S3 − mean(r_S1, r_S2) of one specimen (from its centred S3 contrast: S3 minus the mean of the others)."""
    return float(SEG_CON[spec].loc[g, 'S3'])


def check(rule, g):
    s3 = {spec: s3_vs_early(spec, g) for spec in NAMES}
    if rule == 'mouse: S1/S2 above S3':
        return all(s3[s] <= -EFFECT for s in MOUSE)
    if rule == 'mouse: S3 above S1/S2':
        return all(s3[s] >= EFFECT for s in MOUSE)
    if rule == 'human: low in PT':
        return bool((pt_pairs.loc[g] <= -1).all())
    if rule == 'human: PT-expressed':
        return all(DETECTION[s][g] >= universe_detection[s] for s in HUMAN)
    raise KeyError(rule)


RULES = {'F13b': ['mouse: S1/S2 above S3', 'human: low in PT'], 'Cela1': ['mouse: S1/S2 above S3', 'human: low in PT'],
         'Acsm1': ['mouse: S3 above S1/S2', 'human: low in PT'], 'Anpep': ['mouse: S3 above S1/S2', 'human: PT-expressed'],
         'S100g': ['mouse: S1/S2 above S3', 'human: low in PT'], 'Slc34a1': ['mouse: S1/S2 above S3', 'human: PT-expressed'],
         'Slc34a3': ['human: PT-expressed']}
READING.append("C2 'S100g: mouse PT-expressed (convoluted part: S1/S2)' is read as the rule 'mouse: S1/S2 above S3'; "
               "'human: low in PT' as the protocol's level rule (≤ −1 log2 in all 4 pairs).")
hyp_rows = []
for r in GENES_LIT.itertuples():
    g = r.gene
    row = {'family': r.family, 'gene': g, 'source': r.source, 'published': r.published_statement, 'prediction': r.prediction,
           'status': r.status}
    if r.status == 'eligible':
        row.update({f'detection {s}': DETECTION[s][g] for s in NAMES})
        row.update({f'S3 vs S1/S2 {s}': s3_vs_early(s, g) for s in NAMES})
        row.update({f'human − mouse PT {pair}': pt_pairs.loc[g, pair] for pair in PAIRS})
        row.update({'mouse probes': probes.loc[g, 'mouse_probes'], 'human probes': probes.loc[g, 'human_probes']})
        checks = [f"{rule}: {'consistent' if check(rule, g) else 'not consistent'}" for rule in RULES.get(g, [])]
        if g == 'Slc34a3':
            ratio = pt_pairs.loc['Slc34a3'] - pt_pairs.loc['Slc34a1']
            row.update({f'(Slc34a3 − Slc34a1) human − mouse {pair}': ratio[pair] for pair in PAIRS})
            checks.append(f"Slc34a3 − Slc34a1 higher in human (level, probe-exposed): "
                          f"{'consistent' if (ratio >= 0).all() else 'not consistent'}")
        if r.family == 'F':
            rel = pt_pairs.loc[g] - pt_pairs.loc[TRANSPORTERS].median()
            row.update({f'vs transporter median {pair}': rel[pair] for pair in PAIRS})
            checks.append(f"higher in human than the transporter background (level, probe-exposed): "
                          f"{'consistent' if (rel >= 0).all() else 'not consistent'}")
        row['checks'] = '; '.join(checks) if checks else 'descriptive only'
    hyp_rows.append(row)
GENE_HYP = pd.DataFrame(hyp_rows)
display(GENE_HYP[['family', 'gene', 'status', 'checks']])
display(GENE_HYP.set_index('gene')[[c for c in GENE_HYP.columns if c.startswith(('detection', 'S3 vs'))]].round(2))

## 10 · E1 · Does mouse-only zonation follow male androgen programs? (Xiong 2023)

Per segment k and gene, from the centred within-species segment contrasts c:
- **mouse-zonated in Sk**: c ≥ 0.5 in both mice; **conserved**: and human c ≥ 0.25 in both sections;
  **mouse-only**: and human mean c < 0.25.
- **male-biased in Sk**: Xiong PT-Sk (F vs M) p_adj < 0.05 and log2FC ≤ −1; female-biased the mirror.
- **Primary:** gene level, pooled over segments (mouse-only in some segment and conserved in none, against conserved in
  some segment); male-biased in a segment where the gene is mouse-zonated. Mantel–Haenszel over quintiles of the
  gene's mean mouse PT rate; one-sided p for OR > 1. Per-segment ORs and the female-biased OR are descriptive.

In [ ]:
xiong = {seg: pd.read_csv(XIONG_DIR / f'PT-{seg}_fWTvsmWT.csv').drop_duplicates('symbol').set_index('symbol') for seg in SEGMENTS}
male = {seg: set(t.index[(t.p_adj < .05) & (t.log2FC <= -1)]) for seg, t in xiong.items()}
female = {seg: set(t.index[(t.p_adj < .05) & (t.log2FC >= 1)]) for seg, t in xiong.items()}
tested_xiong = set.intersection(*[set(t.index[t.tested.astype(str).str.upper().eq('TRUE')]) for t in xiong.values()])
READING.append(f"E1 uses genes Xiong tested in all three PT segment tables ({len(tested_xiong & set(GENES)):,} universe genes); "
               "'male-biased' is read as F-vs-M log2FC ≤ −1 with p_adj < 0.05 (negative = higher in males, e.g. Cyp7b1). The 15 "
               "label/orientation genes are excluded (their zonation is forced by the labels, protocol R3).")
mouse_rate = pd.DataFrame(RATES['PT'])[MOUSE].mean(axis=1)
zrows = []
for g in GENES:
    if g not in tested_xiong or g in LABEL_ORIENTATION:
        continue
    row = {'gene': g}
    for seg in SEGMENTS:
        cm = [SEG_CON[s].loc[g, seg] for s in MOUSE]
        ch = [SEG_CON[s].loc[g, seg] for s in HUMAN]
        mz = min(cm) >= .5
        row[f'mouse-zonated {seg}'] = mz
        row[f'class {seg}'] = ('conserved' if mz and min(ch) >= EFFECT else 'mouse-only' if mz and np.mean(ch) < EFFECT
                               else 'other' if mz else '')
        row[f'male {seg}'], row[f'female {seg}'] = g in male[seg], g in female[seg]
    classes_g = {row[f'class {seg}'] for seg in SEGMENTS}
    row['class'] = 'mouse-only' if 'mouse-only' in classes_g and 'conserved' not in classes_g else (
        'conserved' if 'conserved' in classes_g else '')
    zseg = [seg for seg in SEGMENTS if row[f'mouse-zonated {seg}']]
    row['male-biased where zonated'] = any(row[f'male {seg}'] for seg in zseg)
    row['female-biased where zonated'] = any(row[f'female {seg}'] for seg in zseg)
    row['mouse rate'] = mouse_rate[g]
    zrows.append(row)
ZG = pd.DataFrame(zrows).set_index('gene')
ZG['stratum'] = pd.qcut(ZG['mouse rate'], 5, labels=False)
sex_rows = []
for scope in ['pooled', *SEGMENTS]:
    if scope == 'pooled':
        sub = ZG[ZG['class'].isin(['mouse-only', 'conserved'])]
        exposed, cls = sub['class'].eq('mouse-only'), sub['class']
        outcomes = {'male': sub['male-biased where zonated'], 'female': sub['female-biased where zonated']}
    else:
        sub = ZG[ZG[f'class {scope}'].isin(['mouse-only', 'conserved'])]
        exposed = sub[f'class {scope}'].eq('mouse-only')
        outcomes = {'male': sub[f'male {scope}'], 'female': sub[f'female {scope}']}
    for sex, outcome in outcomes.items():
        res = pf.mantel_haenszel(exposed.to_numpy(), outcome.to_numpy(), sub['stratum'].to_numpy())
        sex_rows.append({'scope': scope, 'bias': sex, 'mouse-only genes': int(exposed.sum()),
                         'conserved genes': int((~exposed).sum()),
                         'share biased, mouse-only': float(outcome[exposed].mean()) if exposed.any() else np.nan,
                         'share biased, conserved': float(outcome[~exposed].mean()) if (~exposed).any() else np.nan,
                         'OR (MH)': res['or_mh'], 'p (OR > 1)': res['p'],
                         'role': 'primary' if (scope == 'pooled' and sex == 'male') else 'descriptive'})
SEX_BIAS = pd.DataFrame(sex_rows)
display(SEX_BIAS.round(4))

## 11 · G · Continuous description along the axis (descriptive)

**Curves.** 71's NB specimen curves. A member is usable in a specimen when converged, not separated and detected in
≥ 5% of that specimen's units; its curve is centred by its mean over the grid (log2). The program curve is the median
over usable members (≥ 5). Positions are in segment units (S1 0–1, S2 1–2, S3 2–3; 72's pooled transitions).

**Descriptors per specimen.** flat (amplitude < 0.25) → interior peak/trough → monotone rising/falling (onset = half
change, width = 10–90% span; sharp switch ≤ 0.5, gradient ≥ 1.0) → mixed. A species statement needs both specimens
to agree (same class, positions within 0.25); a human position inside S2 is flagged (weak cell). Across species a
position difference below the registration gap (0.37) is "same place within registration error".

In [ ]:
COND = MODELS['condition']
USABLE = {spec: (COND[f'converged|{spec}'].astype(bool) & ~COND[f'separated|{spec}'].astype(bool)
                 & (DETECTION[spec].to_numpy() >= MIN_DETECTION)) for spec in NAMES}
GIDX = {g: i for i, g in enumerate(GENES)}


REFERENCE = {spec: pf.reference_curve(COND[f'curve|{spec}'], USABLE[spec]) for spec in NAMES}
READING.append('Deviation (recorded after the first run): every within-species curve is also centred by its specimen\'s '
               'median usable gene at each grid point (pf.reference_curve; the within-species analogue of 76/79\'s '
               'centre_rows). Without it, size-matched random gene sets reproduced the human mid-S1 trough and the '
               'S3 0.5–0.75 peak that most programs showed, so those shapes were shared, not program-specific.')


def specimen_program_curve(members, spec):
    idx = np.array([GIDX[g] for g in members])
    return pf.program_curve(COND[f'curve|{spec}'][idx], USABLE[spec][idx], reference=REFERENCE[spec],
                            min_members=MIN_MEMBERS)


def gene_curve(g, spec):
    i = GIDX[g]
    if not USABLE[spec][i]:
        return np.full(len(GRID), np.nan)
    y = COND[f'curve|{spec}'][i]
    return (y - y.mean() - REFERENCE[spec]) / np.log(2)


ALL_PROGRAMS = {**{p: P79[p] for p in META79.index if META79.loc[p, 'role'] == 'confirmatory'}, **PLIT}
SOURCE_OF = {**{p: str(META79.loc[p, 'source']).split(' (')[0] for p in P79}, **{p: str(METALIT.loc[p, 'source']).split(' (')[0] for p in PLIT}}
CURVES_SPEC, desc_rows, species_rows = {}, [], []


def describe(name, curves_by_spec, kind):
    descs = {spec: pf.shape_descriptors(curves_by_spec[spec], S_GRID) for spec in NAMES}
    for spec, d in descs.items():
        desc_rows.append({'name': name, 'kind': kind, 'specimen': spec, 'species': SPECIES_OF[spec], **d,
                          'position (segment)': pf.segment_position(d['position'])})
    stmt = {'mouse': pf.species_statement(descs[MOUSE[0]], descs[MOUSE[1]], weak=None),
            'human': pf.species_statement(descs[HUMAN[0]], descs[HUMAN[1]])}
    row = {'name': name, 'kind': kind}
    for sp, s in stmt.items():
        row.update({f'{sp} shape': s['shape'], f'{sp} position': s['position'],
                    f'{sp} position (segment)': pf.segment_position(s['position']), f'{sp} width': s['width'],
                    f'{sp} switch': s['switch'], f'{sp} flag': s['flag']})
    row['across species'] = pf.cross_species(stmt['mouse'], stmt['human'], gap=REGISTRATION_GAP)
    species_rows.append(row)


for p, members in ALL_PROGRAMS.items():
    curves = {}
    for spec in NAMES:
        curves[spec], n = specimen_program_curve(members, spec)
        CURVES_SPEC[(p, spec)] = curves[spec]
    describe(p, curves, 'program')
for g in GENES_LIT.gene[GENES_LIT.status.eq('eligible')]:
    curves = {spec: gene_curve(g, spec) for spec in NAMES}
    for spec in NAMES:
        CURVES_SPEC[(g, spec)] = curves[spec]
    describe(g, curves, 'single gene')
DESCRIPTORS = pd.DataFrame(desc_rows)
DESC_SPECIES = pd.DataFrame(species_rows).set_index('name')

# Post hoc (recorded): a size-matched random-program null for every agreed non-flat species statement.
rng = np.random.default_rng(81)
POOL = [g for g in GENES if all(USABLE[s][GIDX[g]] for s in NAMES)]
N_NULL = 200


def null_match(members, stmt_mouse, stmt_human):
    """Share of size-matched random programs whose species statement has the same shape and a position within 0.25."""
    hits = {'mouse': 0, 'human': 0}
    for _ in range(N_NULL):
        draw = list(rng.choice(POOL, len(members), replace=False))
        descs = {spec: pf.shape_descriptors(specimen_program_curve(draw, spec)[0], S_GRID) for spec in NAMES}
        for sp, specs, target in (('mouse', MOUSE, stmt_mouse), ('human', HUMAN, stmt_human)):
            s = pf.species_statement(descs[specs[0]], descs[specs[1]], weak=None)
            if target['agree'] and target['shape'] not in ('flat', 'mixed') and s['agree'] and s['shape'] == target['shape'] \
                    and abs(s['position'] - target['position']) <= EFFECT:
                hits[sp] += 1
    return {sp: hits[sp] / N_NULL for sp in hits}


for name in DESC_SPECIES.index[DESC_SPECIES.kind.eq('program')]:
    stmt = {sp: {'agree': np.isfinite(DESC_SPECIES.loc[name, f'{sp} position']),
                 'shape': DESC_SPECIES.loc[name, f'{sp} shape'], 'position': DESC_SPECIES.loc[name, f'{sp} position']}
            for sp in ('mouse', 'human')}
    if not any(s['agree'] and s['shape'] not in ('flat', 'mixed') for s in stmt.values()):
        continue
    rate = null_match(ALL_PROGRAMS[name], stmt['mouse'], stmt['human'])
    for sp in ('mouse', 'human'):
        if stmt[sp]['agree'] and stmt[sp]['shape'] not in ('flat', 'mixed'):
            DESC_SPECIES.loc[name, f'{sp} null match'] = rate[sp]
READING.append(f'Post hoc (recorded): each agreed non-flat program statement is compared with {N_NULL} size-matched '
               'random programs drawn from genes usable in all four specimens (seed 81); "null match" is the share whose '
               'statement has the same shape and a position within 0.25. A statement is called program-specific when '
               'the null match is ≤ 0.05.')
display(DESC_SPECIES[['kind', 'mouse shape', 'mouse position (segment)', 'mouse switch', 'human shape',
                      'human position (segment)', 'human switch', 'human flag', 'across species']
                     + [c for c in ('mouse null match', 'human null match') if c in DESC_SPECIES]])

**The species difference along the axis.** For the A1, A3 and C1 programs: 79's program curve rule on 71's Δ(x)
(centred by the median gene per grid point, signed by d), described with the same descriptors on the condition curve
and on each of the 4 pair curves.

In [ ]:
from pseudospace.aki_along_pt import centre_rows, program_profile

ok_cond = COND['stat|converged'].astype(bool) & ~COND['stat|separated'].astype(bool)
DELTA = {'condition': (centre_rows(COND['delta'], ok_cond)[0], ok_cond)}
for c in CASE:
    for r in REF:
        ok_p = (ok_cond & COND[f'converged|{c}'].astype(bool) & COND[f'converged|{r}'].astype(bool)
                & ~COND[f'separated|{c}'].astype(bool) & ~COND[f'separated|{r}'].astype(bool))
        diff = COND[f'curve|{c}'] - COND[f'curve|{r}']
        ref_ok = ok_p & np.isfinite(diff).all(axis=1)
        DELTA[f'{c}−{r}'] = (diff - np.nanmedian(diff[ref_ok], axis=0), ok_p)
SIGNS_DIFF = {**{p: SIGN_LIT[p] for p in {**A1, **C1}}, **A3_SIGN}
diff_rows, DIFF_CURVES = [], {}
for p, d in SIGNS_DIFF.items():
    descs = {}
    for name, (centred, ok) in DELTA.items():
        prof, n = program_profile(centred, GENES, PLIT[p], usable=ok)
        DIFF_CURVES[(p, name)] = d * prof
        descs[name] = pf.shape_descriptors(d * prof, S_GRID)
    c = descs['condition']
    same = sum(descs[pair]['shape'] == c['shape'] for pair in PAIRS)
    close = sum(descs[pair]['shape'] == c['shape'] and abs(descs[pair]['position'] - c['position']) <= EFFECT
                for pair in PAIRS) if np.isfinite(c['position']) else np.nan
    diff_rows.append({'program': p, 'd': d, 'shape': c['shape'], 'position': c['position'],
                      'position (segment)': pf.segment_position(c['position']), 'width': c['width'], 'switch': c['switch'],
                      'amplitude': c['amplitude'], 'pairs with the same shape': same, 'pairs within 0.25': close})
DIFF_DESC = pd.DataFrame(diff_rows).set_index('program')
# Post hoc (recorded): size-matched random programs on the same centred Δ give interior extrema in about half of draws,
# clustered in S1 and at S3 0.5–0.75 (curve noise is largest there). A difference-curve shape is therefore reported
# only when its amplitude exceeds the 95th percentile of 300 size-matched random programs and ≥ 3 of 4 pairs share it.
rng_d = np.random.default_rng(82)
POOL_D = [g for g, o in zip(GENES, DELTA['condition'][1]) if o]
for p in DIFF_DESC.index:
    amps = []
    for _ in range(300):
        draw = list(rng_d.choice(POOL_D, len(PLIT[p]), replace=False))
        prof, _ = program_profile(DELTA['condition'][0], GENES, draw, usable=DELTA['condition'][1])
        amps.append(np.ptp(prof))
    q95 = float(np.quantile(amps, .95))
    DIFF_DESC.loc[p, 'null amplitude 95%'] = q95
    DIFF_DESC.loc[p, 'reportable'] = bool(DIFF_DESC.loc[p, 'amplitude'] > q95 and DIFF_DESC.loc[p, 'pairs with the same shape'] >= 3
                                          and DIFF_DESC.loc[p, 'shape'] not in ('flat', 'mixed'))
READING.append('Post hoc (recorded): a species-difference curve shape is reportable only when its amplitude exceeds the 95th '
               'percentile of 300 size-matched random programs (seed 82) and at least 3 of 4 pairs share its shape; its '
               'within-segment position stays descriptive, because random programs place extrema at the same positions.')
display(DIFF_DESC.round(3))

**Landmark check** (matched set M: package units with an old structure in notebook 53's LRS-Z table). Unit program
score = the mean over usable members of log2((count + 0.5) / library × the median library), centred per gene per
specimen; a least-squares cubic B-spline per specimen (3 interior knots at the coordinate's quartiles on M) on the
package coordinate and on LRS-Z, each in its own segment units. "Agrees" = the same shape class and position within
0.25 per specimen; "landmark-confirmed" = both specimens of a species agree.

In [ ]:
lrsz = pd.read_csv(LRSZ_FILE).set_index('structure_id')
old_ids = UNITS.obs.old_structure_id.reindex(OBS.index).astype(object)
on_m = old_ids.isin(lrsz.index).to_numpy()
LRS = lrsz.position.reindex(old_ids[on_m].astype(str)).to_numpy(float)
M_POS, M_SPEC, M_SEG = POSITION[on_m], SPECIMEN[on_m], np.asarray(INP['segment'])[on_m]
M_COUNTS, M_LIB = COUNTS[on_m], LIBRARY[on_m]
GUARDS.append(f'Landmark set M: {int(on_m.sum()):,} of {len(on_m):,} analysed units have an LRS-Z position')
coords = {'package': M_POS, 'LRS-Z': LRS}
COORD_UNITS = {}
for name, x in coords.items():
    assert np.all((x >= 0) & (x <= 1)), name
    grid = common_grid(x, M_SPEC, 101)
    cuts = sc.mean_transitions(logistic_transitions(x, M_SEG, M_SPEC), NAMES)
    COORD_UNITS[name] = (grid, sc.to_segment_units(grid, cuts), gam_internal_knots(x, 6))
median_lib = float(np.median(M_LIB))


def centred_scores(spec, idx):
    rows = M_SPEC == spec
    y = np.log2((M_COUNTS[rows][:, idx].toarray() + FLOOR) / M_LIB[rows, None] * median_lib).astype(np.float32)
    return y - y.mean(axis=0)


UNIT_TYPICAL = {spec: np.median(centred_scores(spec, np.flatnonzero(USABLE[spec])), axis=1) for spec in NAMES}
READING.append('Deviation (recorded after the first run): the landmark-check unit score subtracts each unit\'s median '
               'usable gene (same centred log2 scale), which removes the per-unit library effect the 0.5 floor carries.')


def unit_scores(members, spec):
    idx = [GIDX[g] for g in members if USABLE[spec][GIDX[g]]]
    if len(idx) < MIN_MEMBERS:
        return None
    return centred_scores(spec, idx).mean(axis=1) - UNIT_TYPICAL[spec]


land_rows = []
for p, members in ALL_PROGRAMS.items():
    per = {}
    for name, x in coords.items():
        grid, units, knots = COORD_UNITS[name]
        for spec in NAMES:
            score = unit_scores(members, spec)
            if score is None:
                per[(name, spec)] = pf.shape_descriptors(np.full(len(grid), np.nan), units)
                continue
            rows = M_SPEC == spec
            curve = specimen_curves(score[:, None], x[rows], np.repeat(spec, rows.sum()), grid, knots)[spec][0]
            per[(name, spec)] = pf.shape_descriptors(curve, units)
    for sp, specs in (('mouse', MOUSE), ('human', HUMAN)):
        agree = []
        for spec in specs:
            a, b = per[('package', spec)], per[('LRS-Z', spec)]
            ok = a['shape'] == b['shape'] and (a['shape'] in ('flat', 'mixed') or abs(a['position'] - b['position']) <= EFFECT)
            agree.append(ok)
        land_rows.append({'name': p, 'species': sp, **{f'{spec} package': per[('package', spec)]['shape'] + (
            f" {per[('package', spec)]['position']:.2f}" if np.isfinite(per[('package', spec)]['position']) else '') for spec in specs},
            **{f'{spec} LRS-Z': per[('LRS-Z', spec)]['shape'] + (
                f" {per[('LRS-Z', spec)]['position']:.2f}" if np.isfinite(per[('LRS-Z', spec)]['position']) else '') for spec in specs},
            'specimens agreeing across coordinates': int(sum(agree)), 'landmark-confirmed': bool(all(agree))})
LANDMARK = pd.DataFrame(land_rows)
for sp in ('mouse', 'human'):
    conf = LANDMARK[LANDMARK.species.eq(sp)].set_index('name')['landmark-confirmed']
    DESC_SPECIES[f'{sp} landmark-confirmed'] = conf.reindex(DESC_SPECIES.index)
for sp in ('mouse', 'human'):
    positioned = (DESC_SPECIES[f'{sp} position'].notna()
                  & ~DESC_SPECIES[f'{sp} shape'].isin(['flat', 'mixed']) & DESC_SPECIES.kind.eq('program'))
    null_ok = DESC_SPECIES.get(f'{sp} null match', pd.Series(np.nan, index=DESC_SPECIES.index)).le(.05)
    DESC_SPECIES[f'{sp} position confirmed'] = positioned & null_ok & DESC_SPECIES[f'{sp} landmark-confirmed'].eq(True)
CONFIRMED = pd.concat([DESC_SPECIES[DESC_SPECIES[f'{sp} position confirmed']][[f'{sp} shape', f'{sp} position (segment)',
                                                                               f'{sp} switch', 'across species']]
                       .rename(columns=lambda c: c.replace(f'{sp} ', '')).assign(species=sp) for sp in ('mouse', 'human')])
READING.append("A species position is 'confirmed' when both specimens agree, the shape is not flat or mixed, the "
               'size-matched null match is ≤ 0.05 and both specimens agree between the package coordinate and LRS-Z.')
display(LANDMARK.groupby('species')['landmark-confirmed'].agg(['sum', 'count']))
print('Confirmed descriptive positions (specimen-agreed, program-specific, landmark-confirmed):')
display(CONFIRMED)

## 12 · L · Pathway labels (no tests)

A gene group's label is the library term (Hallmark 2020, KEGG 2019 Mouse, Reactome 2022; ties in that order) holding
most of its genes (≥ 3), with the genes listed.

In [ ]:
LIBS = {lib: {k.split('::', 1)[1]: v for k, v in SETS.items() if k.startswith(lib + '::')} for lib in LIBRARY_ORDER}
assert sum(len(v) for v in LIBS.values()) == len(SETS)
groups = {}
for (block, cls), sub in GENE_CLASSES.groupby(['block', 'class']):
    if cls == 'restricted to the listed segment':
        groups[f'A2 {block} restricted'] = sub.gene.tolist()
for cls in ('mouse-only', 'conserved'):
    for biased in (True, False):
        sub = ZG[ZG['class'].eq(cls) & ZG['male-biased where zonated'].eq(biased)]
        if cls == 'conserved' and biased:
            continue
        groups[f"E1 {cls}{' × male-biased' if biased else (' × not male-biased' if cls == 'mouse-only' else '')}"] = sub.index.tolist()
label_rows = []
for name, genes in groups.items():
    lab = pf.pathway_label(genes, LIBS)
    label_rows.append({'group': name, 'genes': len(genes), 'library': lab[0] if lab else '', 'label': lab[1] if lab else '',
                       'genes in label': ', '.join(lab[2]) if lab else '', 'all genes': ', '.join(sorted(genes)[:60])})
LABELS = pd.DataFrame(label_rows)
display(LABELS[['group', 'genes', 'library', 'label', 'genes in label']])

## 13 · Figures

- **81a** A1: the unique and core CellSpectra programs × segment, coloured by Q (log2).
- **81b** Within-species program curves (each specimen) for the A3, C1 and 79 zonation programs.
- **81c** The single-gene hypotheses: each specimen's centred segment contrast.
- **81d** E1: share male-biased among conserved and mouse-only zonated genes.

In [ ]:
from matplotlib.colors import TwoSlopeNorm


def a1_figure():
    progs = list(A1_CLASSES.index)
    q = np.array([[A1_SUMMARY.loc[(p, 'condition'), s] * SIGN_LIT[p] for s in SEGMENTS] for p in progs])
    pairs_q = {p: np.array([[A1_SUMMARY.loc[(p, pair), s] * SIGN_LIT[p] for s in SEGMENTS] for pair in PAIRS]) for p in progs}
    fig, ax = plt.subplots(figsize=(60 * MM, (12 + 6 * len(progs)) * MM))
    im = ax.imshow(np.clip(q, -3, 3), cmap='RdBu_r', norm=TwoSlopeNorm(0, -3, 3), aspect='auto')
    for i, p in enumerate(progs):
        for k in range(3):
            ax.scatter(k + np.linspace(-.3, .3, 4), [i + .32] * 4, s=2,
                       c=[INK if v >= EFFECT else LIGHT for v in pairs_q[p][:, k]], zorder=3)
        if SCOPE_LIT[p] in SEGMENTS:
            ax.add_patch(plt.Rectangle((SEGMENTS.index(SCOPE_LIT[p]) - .5, i - .5), 1, 1, fill=False, ec=INK, lw=1))
    ax.set_xticks(range(3))
    ax.set_xticklabels(SEGMENTS)
    ax.set_yticks(range(len(progs)))
    ax.set_yticklabels(progs)
    fig.colorbar(im, ax=ax, fraction=.05, pad=.04, label='Q (log2)')
    save(fig, 'fig81a_cellspectra_unique_core')


def curves_figure(programs, stem):
    n = len(programs)
    cols = 6
    rows_ = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows_, cols, figsize=(183 * MM, 28 * MM * rows_), squeeze=False, sharex=True)
    for ax, p in zip(axes.flat, programs):
        for spec in NAMES:
            ax.plot(S_GRID, CURVES_SPEC[(p, spec)], color=SPEC_COLOR[spec], lw=.9)
        for b in (1, 2):
            ax.axvline(b, color=LIGHT, lw=.6, zorder=0)
        ax.axhline(0, color=LIGHT, lw=.5, zorder=0)
        ax.set_title(p, loc='left')
    for ax in axes.flat[n:]:
        ax.axis('off')
    for ax in axes[-1]:
        ax.set_xlabel('segment units')
    for ax in axes[:, 0]:
        ax.set_ylabel('log2, centred')
    fig.tight_layout()
    save(fig, stem)


def genes_figure():
    genes = GENE_HYP.gene[GENE_HYP.status.eq('eligible')].tolist()
    fig, axes = plt.subplots(1, len(genes), figsize=(183 * MM, 32 * MM), sharey=True)
    for ax, g in zip(axes, genes):
        for spec in NAMES:
            ax.plot(range(3), SEG_CON[spec].loc[g, list(SEGMENTS)].to_numpy(float), marker='o', ms=2, color=SPEC_COLOR[spec], lw=.8)
        ax.axhline(0, color=LIGHT, lw=.5, zorder=0)
        ax.set_xticks(range(3))
        ax.set_xticklabels(SEGMENTS)
        ax.set_title(g, loc='left')
    axes[0].set_ylabel('segment − other segments (log2, centred)')
    fig.tight_layout()
    save(fig, 'fig81c_gene_hypotheses')


def sex_figure():
    sub = SEX_BIAS[SEX_BIAS.bias.eq('male')].set_index('scope')
    fig, ax = plt.subplots(figsize=(60 * MM, 40 * MM))
    x = np.arange(len(sub))
    ax.bar(x - .18, sub['share biased, conserved'], .36, color=LIGHT, edgecolor=INK, lw=.5, label='conserved')
    ax.bar(x + .18, sub['share biased, mouse-only'], .36, color=MOUSE_COLOR, label='mouse-only')
    ax.set_xticks(x)
    ax.set_xticklabels(sub.index)
    ax.set_ylabel('share male-biased (Xiong 2023)')
    ax.legend()
    save(fig, 'fig81d_sex_bias')


a1_figure()
curves_figure(list(A3) + list(C1) + [p for p in META79.index if META79.loc[p, 'family'] == 'zonation'
                                     and META79.loc[p, 'role'] == 'confirmatory'], 'fig81b_program_curves')
curves_figure(list(A1), 'supp81_cellspectra_unique_curves')
genes_figure()
sex_figure()

## 14 · Summary by paper

In [ ]:
summary_rows = []
for p, r in A1_CLASSES.iterrows():
    summary_rows.append({'paper': 'CellSpectra (Klötzer 2025)', 'item': p, 'published': METALIT.loc[p, 'published_claim'],
                         'ours': f"{r.readout}; {r.localization} (peak {r.peak}, margin {r.margin:.2f}); "
                                 f"P S1/S2/S3 {r['P S1']:.2f}/{r['P S2']:.2f}/{r['P S3']:.2f}"})
for p, r in A3_TABLE.iterrows():
    ds = DESC_SPECIES.loc[p]
    summary_rows.append({'paper': 'CellSpectra (Klötzer 2025)', 'item': p, 'published': METALIT.loc[p, 'published_claim'],
                         'ours': f"{r.readout}; peaks {', '.join(r[f'peak {s}'] for s in NAMES)}; {r.localization}; "
                                 f"mouse {ds['mouse shape']} {ds['mouse position (segment)']}, human {ds['human shape']} "
                                 f"{ds['human position (segment)']} ({ds['across species']})"})
for p, r in C1_CLASSES.iterrows():
    summary_rows.append({'paper': 'Raghubar 2022', 'item': p, 'published': METALIT.loc[p, 'published_claim'],
                         'ours': f"{r.replication} (Q {r['Q at scope']:.2f}, {r['pairs with claimed sign']}/4 pairs); "
                                 f"{r.localization} (peak {r.peak}); detected in human PT "
                                 f"{r['detected ≥10% in both human sections']:.2f} vs {r['universe share detected']:.2f}"})
for r in GENE_HYP.itertuples():
    summary_rows.append({'paper': r.source.split(' (')[0], 'item': r.gene, 'published': r.published,
                         'ours': r.checks if isinstance(r.checks, str) else r.status})
prim = SEX_BIAS[SEX_BIAS.role.eq('primary')].iloc[0]
summary_rows.append({'paper': 'Xiong 2023', 'item': 'E1', 'published': 'mouse PT sex dimorphism, androgen-driven; limited in human',
                     'ours': f"male-biased share {prim['share biased, mouse-only']:.3f} (mouse-only, n {prim['mouse-only genes']}) vs "
                             f"{prim['share biased, conserved']:.3f} (conserved, n {prim['conserved genes']}); "
                             f"OR {prim['OR (MH)']:.2f}, p {prim['p (OR > 1)']:.2g}"})
SUMMARY_BY_PAPER = pd.DataFrame(summary_rows)
display(SUMMARY_BY_PAPER)

## 15 · Outputs

In [ ]:
pd.concat([META79.assign(origin='notebook 79'), METALIT.assign(origin='published-findings')]).to_csv(tables / 'programs.csv')
pd.concat({'A1': A1_TESTS.join(A1_BETWEEN), 'A3': A3_BETWEEN, 'C1': C1_TESTS.join(C1_BETWEEN)}, names=['family']).to_csv(
    tables / 'program_tests.csv')
pd.concat({'A1': A1_CLASSES, 'A3': A3_CLASSES.join(A3_TABLE[['readout']]), 'C1': C1_CLASSES}, names=['family']).to_csv(
    tables / 'classes.csv')
A3_TABLE.to_csv(tables / 'a3_placement.csv')
GENE_CLASSES.to_csv(tables / 'gene_classes_cellspectra.csv', index=False)
A2_COUNTS.to_csv(tables / 'gene_classes_cellspectra_counts.csv')
GENE_HYP.to_csv(tables / 'gene_hypotheses.csv', index=False)
SEX_BIAS.to_csv(tables / 'sex_bias.csv', index=False)
ZG.to_csv(tables / 'sex_bias_genes.csv')
DESCRIPTORS.to_csv(tables / 'descriptors.csv', index=False)
DESC_SPECIES.to_csv(tables / 'descriptors_species.csv')
CONFIRMED.to_csv(tables / 'descriptors_confirmed.csv')
DIFF_DESC.to_csv(tables / 'descriptors_difference.csv')
LANDMARK.to_csv(tables / 'landmark_check.csv', index=False)
LABELS.to_csv(tables / 'labels.csv', index=False)
SUMMARY_BY_PAPER.to_csv(tables / 'summary_by_paper.csv', index=False)
np.savez_compressed(tables / 'curves.npz', grid=GRID, s=S_GRID,
                    **{f'{k[0]}|{k[1]}': v for k, v in CURVES_SPEC.items()},
                    **{f'difference|{k[0]}|{k[1]}': v for k, v in DIFF_CURVES.items()})

## 16 · Readout against the protocol's "adds nothing" rules

In [ ]:
unique_ids = [p for p in A1_CLASSES.index if SCOPE_LIT[p] in SEGMENTS]
a1_any = A1_CLASSES.loc[unique_ids, 'readout'].eq('supported (segment-specific)').any()
g_any = {sp: bool(DESC_SPECIES[f'{sp} position confirmed'].any()) for sp in ('mouse', 'human')}
print(f"A1: {int(A1_CLASSES.loc[unique_ids, 'readout'].eq('supported (segment-specific)').sum())} of {len(unique_ids)} unique "
      f"programs supported as segment-specific → the 'adds nothing' rule is {'not met' if a1_any else 'met'}")
print(f"G: confirmed descriptive positions, mouse {g_any['mouse']}, human {g_any['human']} → the rule is "
      f"{'not met' if any(g_any.values()) else 'met'}")
print('B2b (the microdissection test) is in notebook 82.')

## 17 · How the protocol was read

In [ ]:
READING.extend([
    'Programs come from the committed docs/results/known-programs/ and docs/results/published-findings/ files, with sha256 asserted.',
    "A1 replication (level, PT or the listed segment) is reported with 79's machinery as a descriptive column; its BH runs over the 8 A1 programs.",
    "A1 'peak in another segment by ≥ 0.25' uses 79's margin (Q at the peak minus the next largest Q).",
    'A3: d is the sign of the condition P in the segment with the largest |P|; the within-species placement uses the '
    "centred segment contrasts of 79 §3.4 (program median per specimen).",
    'Gene-level contrasts use the centred segment contrasts (79 §3.4), so a genome-wide shift between segments cancels.',
    'G: the program curve uses each specimen\'s own NB curve from 71 (no refit); the grid is 71\'s common support, so '
    'every specimen curve covers the same positions.',
    'G: the species difference curves follow 79\'s rule (centred Δ, signed by d); pairs agreeing = the same shape class '
    '(and position within 0.25 where a position exists).',
    'Landmark check: LRS-Z positions are notebook 53\'s (zone units / 3); each coordinate\'s segment units come from the '
    'logistic transitions of the package labels on set M on that coordinate.',
    "E1: per segment a gene is 'other' when mouse-zonated but neither conserved nor mouse-only; such genes are excluded.",
    'No stage cache is used: nothing here fits a model; the calibration runs (about 25 cameraPR calls) are recomputed.',
])
HOW = pd.Series(READING, name='how the protocol was read')
HOW.to_csv(tables / 'how_the_protocol_was_read.csv', index=False)
for line in READING:
    print('·', line)

## 18 · Run record

In [ ]:
record = {'notebook_logic_version': NOTEBOOK_LOGIC_VERSION, 'protocol': 'docs/results/pt-published-findings.md (commit 0a44dec)',
          'pass1_fingerprint': FINGERPRINT, 'sha256': {p.name: v for p, v in SHA.items()},
          'units': int(len(POSITION)), 'genes': len(GENES), 'library_sets': len(SETS), 'guards': GUARDS,
          'calibration': CALIBRATION.reset_index().to_dict('records'),
          'seconds': round(time.perf_counter() - NOTEBOOK_START, 1)}
(output / 'run_record.json').write_text(json.dumps(record, indent=1, default=str))
print(f"Done in {record['seconds']:.0f} s")